In [2]:
# 1. Check GPU and Python environment
import os, platform, subprocess, sys

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', repr(exc))

subprocess.run(['nvidia-smi'], check=False)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Torch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [5]:
# 2. Get the project code
from pathlib import Path
import shutil

USE_GITHUB = True
REPO_URL = 'https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git'
REPO_BRANCH = 'branch-h'
PROJECT_DIR = Path('/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring')

if USE_GITHUB:
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No zip uploaded.')
    zip_name = next(iter(uploaded.keys()))
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(PROJECT_DIR)
    nested = [p for p in PROJECT_DIR.iterdir() if p.is_dir() and (p / 'src').exists()]
    if nested:
        PROJECT_DIR = nested[0]

os.chdir(PROJECT_DIR)
print('Project dir:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir())[:20])

Project dir: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
Files: ['.git', '.gitignore', 'README.md', 'benchmarks', 'context.md', 'data', 'environment.yml', 'notebooks', 'pyproject.toml', 'requirements.txt', 'scripts', 'specifications.md', 'src', 'tdc_kv_results_experiment_plan.md', 'tests']


In [6]:
from pathlib import Path
import subprocess
import sys
import os

print("Working directory:", Path.cwd())

subprocess.run(["git", "branch", "--show-current"])
subprocess.run(["git", "log", "-1", "--oneline"])

test_file = Path("tests/test_hf_cache_e2e.py")
print("Test file exists:", test_file.exists())
print("Test file:", test_file.resolve())

Working directory: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
Test file exists: True
Test file: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/tests/test_hf_cache_e2e.py


In [7]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchvision",
        "torchaudio",
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)



Return code: 0


In [8]:
import importlib.util
import torch
import transformers

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print(
    "Torchvision installed:",
    importlib.util.find_spec("torchvision") is not None,
)
print(
    "TorchAudio installed:",
    importlib.util.find_spec("torchaudio") is not None,
)

from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    LlamaConfig,
    LlamaForCausalLM,
    Qwen2Config,
    Qwen2ForCausalLM,
)

print("GPT-2 import: OK")
print("Llama import: OK")
print("Qwen2 import: OK")

PyTorch: 2.13.0+cu130
PyTorch CUDA: 13.0
Transformers: 5.14.1
Torchvision installed: False
TorchAudio installed: False
GPT-2 import: OK
Llama import: OK
Qwen2 import: OK


In [9]:
import subprocess
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_hf_cache_e2e.py",
        "-x",
        "-vv",
        "--tb=long",
    ],
    text=True,
    capture_output=True,
)

print("\nRETURN CODE:", result.returncode)
print("\nSTDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.13.0+cu130
Transformers: 5.14.1
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB

RETURN CODE: 1

STDOUT:
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
configfile: pyproject.toml
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collecting ... collected 6 items

tests/test_hf_cache_e2e.py::test_compressed_cache_logits_match_independent_forward[gpt2] FAILED [ 16%]

=================================== FAILURES ===================================
_________ test_compressed_cache_logits_match_independent_forward[gpt2] _________

family = 'gpt2'

    @pytest.mark.parametrize("family", ["gpt2", "llama", "qwen2"])
    def test_compressed_cache_logits_match_independent_forward(family):
        model = _model_f

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True,
)

print("Return code:", test_result.returncode)

if test_result.returncode != 0:
    raise RuntimeError("The full test suite failed.")